In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

from pyspark.sql.functions import (
    col,
    sum,
    avg,
    count,
    when
)

In [2]:
spark = (
    SparkSession.builder
    .appName("Practical_Review_Sales_Analytics")
    .master("local[4]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 4.0.4


In [3]:
raw_sales_data = [
    (1, "Laptop", "Electronics", 2, 55000.0),
    (2, "Mouse", "Electronics", 5, 800.0),
    (3, "Keyboard", "Electronics", 3, 1500.0),
    (4, "Chair", "Furniture", 2, 4500.0),
    (5, "Desk", "Furniture", 1, 8000.0),
    (6, "Monitor", "Electronics", 2, 12000.0),
    (7, "Notebook", "Stationery", 10, 100.0),
    (8, "Pen", "Stationery", 20, 30.0),
    (9, "Headphones", "Electronics", 4, 2500.0),
    (10, "Table", "Furniture", 2, 7000.0)
]

print("Number of raw records:", len(raw_sales_data))

Number of raw records: 10


In [4]:
sales_schema = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("product", StringType(), False),
    StructField("category", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False)
])

In [5]:
sales_df = spark.createDataFrame(
    raw_sales_data,
    schema=sales_schema
)

print("RAW DATAFRAME")
sales_df.show()

RAW DATAFRAME
+-------+----------+-----------+--------+----------+
|sale_id|   product|   category|quantity|unit_price|
+-------+----------+-----------+--------+----------+
|      1|    Laptop|Electronics|       2|   55000.0|
|      2|     Mouse|Electronics|       5|     800.0|
|      3|  Keyboard|Electronics|       3|    1500.0|
|      4|     Chair|  Furniture|       2|    4500.0|
|      5|      Desk|  Furniture|       1|    8000.0|
|      6|   Monitor|Electronics|       2|   12000.0|
|      7|  Notebook| Stationery|      10|     100.0|
|      8|       Pen| Stationery|      20|      30.0|
|      9|Headphones|Electronics|       4|    2500.0|
|     10|     Table|  Furniture|       2|    7000.0|
+-------+----------+-----------+--------+----------+



In [6]:
sales_df.printSchema()

root
 |-- sale_id: integer (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: double (nullable = false)



In [7]:
print("Null value validation")

sales_df.select(
    count(when(col("sale_id").isNull(), True)).alias("null_sale_id"),
    count(when(col("product").isNull(), True)).alias("null_product"),
    count(when(col("category").isNull(), True)).alias("null_category"),
    count(when(col("quantity").isNull(), True)).alias("null_quantity"),
    count(when(col("unit_price").isNull(), True)).alias("null_unit_price")
).show()

Null value validation
+------------+------------+-------------+-------------+---------------+
|null_sale_id|null_product|null_category|null_quantity|null_unit_price|
+------------+------------+-------------+-------------+---------------+
|           0|           0|            0|            0|              0|
+------------+------------+-------------+-------------+---------------+



In [8]:
invalid_quantity = sales_df.filter(
    col("quantity") <= 0
).count()

print("Invalid quantity records:", invalid_quantity)

Invalid quantity records: 0


In [10]:
invalid_price = sales_df.filter(
    col("unit_price") <= 0
).count()

print("Invalid price records:", invalid_price)

Invalid price records: 0


In [11]:
if invalid_quantity > 0 or invalid_price > 0:
    raise ValueError("Data validation failed.")

print("Data validation passed.")

Data validation passed.


In [12]:
transformed_df = sales_df.withColumn(
    "total_amount",
    col("quantity") * col("unit_price")
)

print("TRANSFORMED DATA")
transformed_df.show()

TRANSFORMED DATA
+-------+----------+-----------+--------+----------+------------+
|sale_id|   product|   category|quantity|unit_price|total_amount|
+-------+----------+-----------+--------+----------+------------+
|      1|    Laptop|Electronics|       2|   55000.0|    110000.0|
|      2|     Mouse|Electronics|       5|     800.0|      4000.0|
|      3|  Keyboard|Electronics|       3|    1500.0|      4500.0|
|      4|     Chair|  Furniture|       2|    4500.0|      9000.0|
|      5|      Desk|  Furniture|       1|    8000.0|      8000.0|
|      6|   Monitor|Electronics|       2|   12000.0|     24000.0|
|      7|  Notebook| Stationery|      10|     100.0|      1000.0|
|      8|       Pen| Stationery|      20|      30.0|       600.0|
|      9|Headphones|Electronics|       4|    2500.0|     10000.0|
|     10|     Table|  Furniture|       2|    7000.0|     14000.0|
+-------+----------+-----------+--------+----------+------------+



In [13]:
category_analysis = (
    transformed_df
    .groupBy("category")
    .agg(
        sum("total_amount").alias("total_revenue"),
        sum("quantity").alias("total_quantity"),
        count("sale_id").alias("number_of_sales"),
        avg("unit_price").alias("average_unit_price")
    )
    .orderBy(
        col("total_revenue").desc()
    )
)

In [14]:
print("CATEGORY-LEVEL ANALYTICAL OUTPUT")

category_analysis.show()

CATEGORY-LEVEL ANALYTICAL OUTPUT
+-----------+-------------+--------------+---------------+------------------+
|   category|total_revenue|total_quantity|number_of_sales|average_unit_price|
+-----------+-------------+--------------+---------------+------------------+
|Electronics|     152500.0|            16|              5|           14360.0|
|  Furniture|      31000.0|             5|              3|            6500.0|
| Stationery|       1600.0|            30|              2|              65.0|
+-----------+-------------+--------------+---------------+------------------+



In [15]:
product_analysis = (
    transformed_df
    .groupBy("product")
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("total_amount").alias("total_revenue")
    )
    .orderBy(
        col("total_revenue").desc()
    )
)

print("PRODUCT-LEVEL ANALYTICAL OUTPUT")

product_analysis.show()

PRODUCT-LEVEL ANALYTICAL OUTPUT
+----------+--------------+-------------+
|   product|total_quantity|total_revenue|
+----------+--------------+-------------+
|    Laptop|             2|     110000.0|
|   Monitor|             2|      24000.0|
|     Table|             2|      14000.0|
|Headphones|             4|      10000.0|
|     Chair|             2|       9000.0|
|      Desk|             1|       8000.0|
|  Keyboard|             3|       4500.0|
|     Mouse|             5|       4000.0|
|  Notebook|            10|       1000.0|
|       Pen|            20|        600.0|
+----------+--------------+-------------+



In [16]:
category_analysis.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [5]: [sale_id#0, product#1, category#2, quantity#3, unit_price#4]
Arguments: [sale_id#0, product#1, category#2, quantity#3, unit_price#4], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [4]: [category#2, quantity#3, unit_price#4, (cast(quantity#3 as double) * unit_price#4) AS total_amount#65]
Input [5]: [sale_id#0, product#1, category#2, quantity#3, unit_price#4]

(3) HashAggregate
Input [4]: [category#2, quantity#3, unit_price#4, total_amount#65]
Keys [1]: [category#2]
Functions [4]: [partial_sum(total_amount#65), partial_sum(quantity#3), partial_count(1), partial_avg(unit_price#4)]
Aggregate Attributes [5]: [sum#104, sum#105L, count#155L, 

In [17]:
print("FINAL ANALYTICAL RESULT")

category_analysis.show(truncate=False)

FINAL ANALYTICAL RESULT
+-----------+-------------+--------------+---------------+------------------+
|category   |total_revenue|total_quantity|number_of_sales|average_unit_price|
+-----------+-------------+--------------+---------------+------------------+
|Electronics|152500.0     |16            |5              |14360.0           |
|Furniture  |31000.0      |5             |3              |6500.0            |
|Stationery |1600.0       |30            |2              |65.0              |
+-----------+-------------+--------------+---------------+------------------+

